# Overview

Author: Chris McCormick

This journal covers the pre-training experiments done at "GPT-2 scale", i.e. around 100M - 117M parameters.

Before getting these running, I had to battle with some issues with the training loss--I put the notes on this in section 2. (They might be less interesting to read than section 1).

# ▂▂▂▂▂▂▂▂▂▂▂▂

# S1. GPT-2 Scale Experiments

Once I resolved the issues with the training loss (see later section), I was able to run some experiments at the full GPT-2 scale (12 layers, model dim 768).

## C4 Dataset

For the larger model, I decided to change up the dataset.

With wikitext103 (103 million articles), we'd been running something like 13 epochs over this training set, which seems like a lot!

I could imagine that we were over-fitting that data, and that this would only get worse with a much larger model.

So I switched to training on 1% of Ai2's C4 dataset (cleaned common crawl corpus or something like that).



**Streaming vs. Pre-Processed**


I didn't try streaming the dataset--I just assume that this would lead to the dataset processing being a bottleneck because of how small our model is (i.e., it can chew through a lot of text fast).

So I had a coding agent write a notebook for pre-processing and saving the data to avoid this.

This was actually a trickier engineering problem than I expected. The Colab instances have an enormous amount of RAM, and 1% of C4 actually sits in memory fine as a list of strings.

However, when I went to turn this into a Dataset object, memory use exploded. Something about it is horribly memory-inefficient. GPT thought it was something to do with the use of `arrow`. 🤷‍♂️ I didn't really dig into it.

The solution was basically to shard the dataset and work on it in chunks, which the Dataset class seems to support well.




## Sequence Length 128

I started with sequence length 128, anticipating that 1,024 would be incredibly slow.

Here are the results. That first row actually corresponds to MHA--the shared space dimensions were set, but the spaces weren't enabled (I've since addressed this so that unused subspaces are always set to size 0).

The table is sorted by perplexity.


<img src='https://lh3.googleusercontent.com/d/1gHHuyMR1fM3lhIVBorKSpB6mB1xeBgHF' alt='Screenshot' width='900' />

MHA performed the best (lowest perplexity), but was slowest. (I think this is the first time that MLA was actually faster!)

After MHA is standard MLA, then MLA with an output subspace. Removing the Query space didn't change it much.

These results are on wandb [here](https://wandb.ai/chrismccormick/decoder-pretrain-c4/table?nw=nwuserchrismccormick). You can filter for the tag `gpt-2_128`.


## Sequence Length 1024

Those length 128 experiments were intended mainly as quick sanity tests. To run more interesting benchmarks, we need to be able to support at least 1,024 tokens, preferably 2,048.



**RoPE Scaling**

To get to 2,048, I ultimately landed on training at length 1,024 with RoPE scaling set to `linear` with a factor of 2.

That means the model would natively support a length of 1,024, and we'd use RoPE scaling to increase that.


**Distractions**

Before moving on to length 1024, I got distracted exploring a number of ideas for reducing compute cost, anticipating how long 1024 was going to take.

I'll share about those in the next section.



**~1.6 → ~3.2 billion tokens**

Larger models need more training data, so I decided to increase our dataset size to 3 billion tokens.

I pre-processed 2% of C4 (which I think is a little more than 3B tokens?).

On GPT's advice, I didn't change the learning rate or batch size despite the increased token count.

We went from:

```
12,500 steps × batch size 1,024 × seq len 128 = 1.634e9 tokens
```

To:

```
3,000 steps × batch size 1,024 × seq len 1024 = 3.15e9 tokens.
```

### Faster on Longer Sequences??



My research into compute-cost strategies was definitely just a distraction, because it turned out that 1,024 was _faster_!

How can we account for this, especially given that I also ~doubled the dataset size?



_Fewer Samples_

For a given dataset size, increasing the sequence length decreases the number of samples needed to see the same number of tokens.

Going from 128 to 1024 means 8x fewer samples.

We went from 12,500 steps at length 128 to 3,000 steps at length 1024


_Flash Attention_

Length 1,024 requires 16x more attention score calculations than length 128.

My understand of Flash Attention is that:

* Attention is highgly memory bound, so the speed is much lower than the GPU's theoretical maximum.
* Flash Attention is able to improve the useage of the compute enough that, up to a point, it can make the calculation time grow linearly instead of quadratically.
    * The only way this is possible is if it's still slower than the theoretical max, but better than the "eager" implementation.



_Attention vs. FFN_

Flash Attention alone can't explain it. We reduced the number of steps by ~4x, but attention should still be taking 8x longer at 1,024.

Maybe the remainder of the explanation is that the GPU is able to evaluate the FFNs more efficiently at length 1,024 than at 128. So much so that 3,000 batches at length 1,024 is actually faster than 12,500 batches at length 128.

Further, I think it would need to be the case that the FFN's are consuming substantially more time than Attention, such that the additional time taken by attention is outweighed by the improved FFN efficiency.

_Conclusion_

I think the above can explain the outcome in theory, but it might be smart to review the code and configurations to see if there isn't some other explanation as well.

I could try running another experiment with the current code, but at length 128 again, just to confirm that it's still taking 2h 45m at the shorter length.


### Results

Below are the results, sorted by increasing perplexity.

The first row, lime green, is standard MHA., still performing the best.

Overall, note how the perplexity follows the model's parameter count (with only one exception, the brown run), with more parameters translating to lower perplexity.

<img src='https://lh3.googleusercontent.com/d/1WT5OG7wNPJXRPORe_PqGlDQVFk6AOoBv' alt='Screenshot' width='900' />

## Distractions

I'm cutting this journal short because it's time to move on, but briefly, here's what I was contemplating:



_Task-focused dataset_

- What if we could train and test faster by honing in on a specific task?
- If the model was trained specifically to do GSM8K, for example, could it train on less data and perform well enough to compare hyperparameter choices?

_Length Curriculum_

* You can reduce training cost by starting out training the model on short sequences, then progressively longer.

_MQA_

* The original MLA uses a trick where it effectively is performing MQA (single key and value) but with a larger head size.
    * I wondered if we might see higher performance if we took the same approach.
* To test this, I needed FlashAttention working to get proper support for MQA / GQA (i.e., to do it fast, not naive copying)
    
_FlashAttention_

It was difficult to get FlashAttention working on Colab, but I did figure it out. There's a GitHub repo that serves up pre-built wheels, and I just needed to find the right one for Colab's combo of CUDA + PyTorch.

Really glad I have that now. I haven't measured carefully, but it seems like FA is faster.



_RoPE on Values_

* Implementing MQA made me wonder if it would work to apply RoPE to the Value vectors as well.
* You can do this by applying the Key RoPE embeddings to the Values, and taking the RoPE embeddings for the query and multiplying them with the value vectors.

(I'll stop there).


# ▂▂▂▂▂▂▂▂▂▂▂▂

# S2. Loss & Grad Norm Problems

While trying to increase the scale of the model to GPT-2-small, I ran into some training issues.

Within a few steps, the training loss hit 0.0, and the grad norm went to NaN.

Debugging this took me all over the place, and I'm not sure what actually fixed it, because I ran into what was actually a second, unrelated issue before everything was working normally.

First, here's the story on the loss crashing.

### Loss Hitting Zero

**Loss Crashing while Scaling Up**

GPT-2 small has dimensions:

* n_layers = 12
* d_model = 768
* d_head = 64, n_heads = 12

We were previously at:

* n_layers = 6
* d_model = 256
* d_head = 32, n_heads = 8

When I tried training at gpt-2 scale, the loss was crashing to 0 immediately.

The first thing I found was that training did work at this scale:

* n_layers = 10
* d_model = 576
* d_head = 48, n_heads = 12

And that as soon as I increased from 10 to 12 layers, I'd run into the loss issue.


**Debugging with ChatGPT**

I took the problem to GPT, and it gave me a whole slew of things to try. Here's what I did based on its suggestions:

* It suggested switching from RMSNorm to LayerNorm, describing it as more forgiving, so I made that a configuration option. This didn't solve it.
* It suggested trying a much smaller learning rate, claiming that my current one was aggressive. This didn't resolve it.
    * I'm glad it wasn't the answer because it would have meant much longer training times!
* It found that I was still using some weight initialization strategies carried over from the Encoder version of the model, and that these were inconsistent with how Decoders typically do it, so we fixed that.

I think the item may be what actually fixed it, but I'd have to backtrack to confirm.

At some point, the problem switched from the loss crashing to 0 to having the loss blow up to, e.g., 400 (and the grad norm to something like 40), so I was still in debugging mode.



### Exploding Loss & Gradients


I'm surprised the AI wasn't more intelligent about this one, because I don't think a loss of 400 is possible. I think the maximum possible loss is base-2 log of the vocabulary size, which for ~56,000 tokens is about 16. This should have pointed us more directly at the problem.

**Tons of Printouts**

Instead, I had Claude write me a fully instrumented version of our code--printing out the norm of the updates to every single weight matrix at each step--so that we could try and identify where the explosion was happening.

Side Note: It's incredible that we can do something like that now. We can write huge amounts of throw-away code, because a coding agent can do it for us.

This didn't reveal the problem! In fact, the loss looked fine in the instrumented version!




**Sneaky Fixes**

The first thing I found was that Claude had made a few changes to the code in addition to adding the print statements, without telling me about them.

Once I realized that, I went through and folded in its fixes.

I think I narrowed it down from there to the fact that it was hooking in a custom "step" function.



**Gradient Accumulation**

At some point, I finally noticed what was going on--the loss and gradient norm were being multiplied by the gradient accumulation steps.

I had the accumulation steps set to 32. If you divide ~400 by 32, you get a reasonable starting Loss value of 12.5.

(At some point I dropped the accumulation steps to 8, and the loss went to ~100).

So somehow the loss wasn't getting divided properly by the HF Trainer, but Claude's custom step function was handling it correctly.

**Custom Loss Function**

Trying to debug this with the AI, it kept proposing that we write a custom step function to handle it correctly.

That seemed like a ridiculous solution--why would that be necessary??

As a sanity check, I tried swapping in a standard model (GPT-2?), and the loss was fine. So that narrowed it down to an issue with our model, and likely an issue with our LM head implementation.

Comparing ours to a couple other models, I noticed that we had more of a custom implementation of the cross entropy loss:

```python
        # Compute loss if labels are provided
        loss = None
        if labels is not None:
            # Shift logits and labels for causal language modeling
            # We predict the next token, so compare logits[..., :-1, :] with labels[..., 1:]
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            # Flatten for cross entropy computation
            # cross_entropy expects [N, C] logits and [N] targets
            batch_size, seq_len = shift_logits.shape[:2]
            vocab_size = shift_logits.shape[-1]
            
            flat_logits = shift_logits.view(-1, vocab_size)
            flat_labels = shift_labels.view(-1)
            
            # Compute cross entropy loss
            # ignore_index=-100 ignores padding tokens in loss computation
            loss = F.cross_entropy(
                flat_logits,
                flat_labels,
                ignore_index=-100
            )
```

Whereas in other models, they just write:

```python
loss = None
if labels is not None:
    # (Chris: This calls a huggingface-defined function built into one of the model classes that you inherit from)
    loss = self.loss_function(logits=logits, labels=labels, vocab_size=self.config.vocab_size, **kwargs)
```

I don't think the implementation in our code was wrong, necessarily, and honestly I don't know why we were suddenly running into this problem.

Maybe it was one of Claude's changes that I copied over from its instrumented version?

At any rate, switching to the built-in HuggingFace function finally resolved it!

I'll share more thoughts on all of this in the "Lessons Learned" section.

# ▂▂▂▂▂▂▂▂▂▂▂▂

# S3. Next Steps

## Tasks

* The speed improvement is still a little surprising / suspicious. Test length 128 again to make sure the speed can't be explained by a code change.
* Fine-tune the length 1024 models on some length 2048 data.
    * (Not covered in this journal, but we currently have some llm-eval results and the model's underperforming, this could be part of it).
* I learned that pre-training likes really large batch sizes, and it seems that there are conventions for relating the batch size to the parameter count.
    * We should understand this better and apply it if needed.
* Evaluate the models with llm-eval
* Use the HF model repo properly, e.g.
    * Create one model repo, but create branches.
    * Include the code with the model (Need to learn about packages and import paths).

**Didn't Cover:**

* We identified at the end of the experiment that much of the increase in perplexity could be attributed to the Key-Value subspace.
    * When we ran with just KV, the perplexity jumped.
    * Re-running with KV space size of 128 helped.




## Questions

RoPE scaling:

1. What can you typically get away with in terms of scaling factor?
2. Why is it that the RoPE scaling needs to be configured prior to pre-training? How is this impacting things?
3. I had forgotten--we need to do a little bit of training at 2,048 I think?



## Side Ideas

Some of the stuff under "Distractions" seems worth exploring further at some point if there's time.